In [ ]:
import os
import sqlite3
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

class SoccerAnalyzer:
    def __init__(self, league_id):
        """
        Initialiseert de verbinding met de database.
        
        Download de meest recente dataset van hogomathien over soccer van kagglehub en slaat de data op in de locale cache.

        Args:
            league_id (int): De ID van de league die geanalyseerd moet worden
        
        Attributes:
            db_path (str): Het pad naar de database
            conn (sqlite3.connect): De verbinding met de database
            league_id (int): De ID van de league
        """
        # Download de dataset
        path = kagglehub.dataset_download("hugomathien/soccer") 
        self.db_path = os.path.join(path, 'database.sqlite')
        self.conn = sqlite3.connect(self.db_path)
        print(f"Verbonden met database op: {self.db_path}")

        self.league_id = league_id

    def get_matches_per_season(self, team_id):
        """
        1A

        Toont het aantal gespeelde wedstrijden per seizoen op voor een specifiek team.
        Berekent met zowel home_team_api_id als away_team_api_id

        Args:
            team_id (int): De ID van het team

        Returns:
            pd.DataFrame: Een DataFrame met de volgende kolommen:
                - 'season': het seizoen
                - 'aantal_wedstrijden': het aantal gespeelde wedstrijden
        """

        query = f"""
        SELECT 
            season, 
            COUNT(*) as aantal_wedstrijden -- Selecteer de seizoenen en het aantal wedstrijden
        FROM Match

        -- Pak alleen de wedstrijden van het gekozen team
        WHERE home_team_api_id = {team_id} OR away_team_api_id = {team_id} 

        -- Groepeer bij het seizoen
        GROUP BY season 
        """

        return pd.read_sql(query, self.conn)
    
    def get_matches_2010(self, team_id):
        """
        1B

        Geeft een overzicht van het aantal gespeelde wedstrijden in 2010.
        Hij berekent specifiek op 2010-01-01 tot 2010-12-31, zowel home_team_api_id als away_team_api_id

        Args:
            team_id (int): De ID van het team

        Returns:
            pd.DataFrame: Een DataFrame met de volgende kolommen:
                - 'season': het seizoen
                - 'aantal_wedstrijden': het aantal gespeelde wedstrijden
        """

        query = f"""
        SELECT 
            season AS Seizoen, 
            COUNT(*) AS aantal_wedstrijden -- Selecteer de seizoenen en het aantal wedstrijden
        FROM Match

        -- Pak alleen de wedstrijden van het gekozen team
        WHERE (home_team_api_id = {team_id} OR away_team_api_id = {team_id})

        -- Pak alleen van het kalenderjaar 2010
        AND date BETWEEN '2010-01-01' AND '2010-12-31' 

        -- Groepeer bij het seizoen
        GROUP BY season 
        """

        return pd.read_sql(query, self.conn) 
    
    def get_league_standings(self):
        """
        1C & 1

        Berekent de eindstand van de competitie per seizoen

        1. Punten worden gegeven voor elke wedstrijd gewonnen, gelijk of verloren
        2. Thuis en uit worden hier beide berekent en samen gevoegd
        3. De totale punten worden uitgerekend per seizoen
        4. Teams worden elk seizoen geordend op aantal punten

        Returns:
            pd.DataFrame: Een DataFrame met de volgende kolommen:
                - 'Seizoen': het seizoen
                - 'team_id': het team id
                - 'Teamnaam': de naam van het team
                - 'Totaal_punten': het aantal punten
                - 'Ranglijst_positie': de eindstand
        """

        query = f"""
        -- Bereken de punten voor elke match
        WITH MatchPoints AS (
            -- Team id als ze thuis spelen
            SELECT 
                season, home_team_api_id AS team_id,

                -- Bereken de punten voor elke match
                CASE WHEN home_team_goal > away_team_goal THEN 3
                     WHEN home_team_goal = away_team_goal THEN 1 ELSE 0 END AS points 

                -- 3 punten voor een win, 1 punt voor een gelijkspel, 0 punten voor een verlies
            
            FROM Match 

            
            WHERE league_id = {self.league_id} 
            

            -- Selecteer beide SELECTS volledig
            UNION ALL 
            
            -- Team id als ze uit spelen
            SELECT 
                season, away_team_api_id AS team_id,

                -- Bereken de punten voor elke match
                CASE WHEN away_team_goal > home_team_goal THEN 3
                     WHEN away_team_goal = home_team_goal THEN 1 ELSE 0 END AS points 

                -- 3 punten voor een win, 1 punt voor een gelijkspel, 0 punten voor een verlies

            FROM Match 

            -- Pak de matches van de gekozen competitie
            WHERE league_id = {self.league_id}
        ) 

        SELECT 
            -- MP = MatchPoints
            MP.season AS Seizoen, 
            MP.team_id,
            -- T = Team
            T.team_long_name AS Teamnaam, 

            -- Bereken de totale punten
            SUM(MP.points) AS Totaal_Punten, 

            -- Per seizoen bereken de ranglijst van de teams
            -- Maakt de ranglijst, RANK() zorgt ervoor dat alles een nummer krijgt met de hoogste 1, dan 2 en 3 (Door de ORDER BY is de ranglijst geordend)
            RANK() OVER (
                -- https://www.sqlshack.com/sql-partition-by-clause-overview/
                -- Maakt een ranglijst per seizoen, maar inplaats van GROUP BY bewaard deze elk team wat in dit seizoen staat
                PARTITION BY MP.season 
                -- Sorteer de teams binnen dat seizoen op aantal punten
                ORDER BY SUM(MP.points) DESC
            ) AS Ranglijst_Positie 

        
        FROM MatchPoints AS MP
        JOIN Team AS T 

        -- Verbind team_api_id met team_id
        ON T.team_api_id = MP.team_id 

        -- Groepeer bij seizoen, het team_id en de teamnaam
        GROUP BY MP.season, MP.team_id, T.team_long_name 
        -- Sorteer de teams binnen dat seizoen op aantal punten
        ORDER BY MP.season, Ranglijst_Positie
        """

        return pd.read_sql(query, self.conn)

    def get_my_team_ranking_per_season(self, all_standings, team_name):
        """
        1D

        Filtert de ranglijst om 1 specifiek team per seizoen te zien (vanuit de functie get_league_standings)

        Args:
            all_standings (pd.DataFrame): De eindstand van de competitie
            team_name (str): De naam van het team

        Returns:
            pd.DataFrame: Een DataFrame met de volgende kolommen:
                - 'Seizoen': het seizoen
                - 'Teamnaam': de naam van het team
                - 'Totaal_punten': het aantal punten
                - 'Ranglijst_positie': de eindstand
        """

        # Pak alle standen van het gekozen team
        my_team_ranking = all_standings[all_standings['Teamnaam'] == team_name]
        
        return my_team_ranking[['Seizoen', 'Teamnaam', 'Totaal_Punten', 'Ranglijst_Positie']]


analyzer = SoccerAnalyzer(21518)
display(analyzer.get_matches_per_season(8634))
display(analyzer.get_matches_2010(8634))
display(analyzer.get_league_standings())
display(analyzer.get_my_team_ranking_per_season(analyzer.get_league_standings(), 'FC Barcelona'))

Verbonden met database op: C:\Users\yorri\.cache\kagglehub\datasets\hugomathien\soccer\versions\10\database.sqlite


,season,aantal_wedstrijden
0,2008/2009,38
1,2009/2010,38
2,2010/2011,38
3,2011/2012,38
4,2012/2013,38
5,2013/2014,38
6,2014/2015,38
7,2015/2016,38


,Seizoen,aantal_wedstrijden
0,2009/2010,23
1,2010/2011,16


,Seizoen,team_id,Teamnaam,Totaal_Punten,Ranglijst_Positie
0,2008/2009,8634,FC Barcelona,87,1
1,2008/2009,8633,Real Madrid CF,78,2
2,2008/2009,8302,Sevilla FC,70,3
3,2008/2009,9906,Atlético Madrid,67,4
4,2008/2009,10205,Villarreal CF,65,5
...,...,...,...,...,...
155,2015/2016,7878,Granada CF,39,16
156,2015/2016,9869,Real Sporting de Gijón,39,16
157,2015/2016,8370,Rayo Vallecano,38,18
158,2015/2016,8305,Getafe CF,36,19


,Seizoen,Teamnaam,Totaal_Punten,Ranglijst_Positie
0,2008/2009,FC Barcelona,87,1
20,2009/2010,FC Barcelona,99,1
40,2010/2011,FC Barcelona,96,1
61,2011/2012,FC Barcelona,91,2
80,2012/2013,FC Barcelona,100,1
102,2013/2014,FC Barcelona,87,2
120,2014/2015,FC Barcelona,94,1
140,2015/2016,FC Barcelona,91,1
